In [ ]:
import json
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


# ============================================================
# PREMIUM STYLE
# ============================================================
plt.style.use("dark_background")

BG = "#0b0f14"
PANEL = "#141a22"
ACCENT = "#00c7ff"
TEXT = "white"


# ============================================================
# LOAD DATA
# ============================================================
BASE_DIR = os.path.abspath("..")
DATASET_DIR = os.path.join(BASE_DIR, "datasets")


def load_json(filename):
    with open(os.path.join(DATASET_DIR, filename), "r", encoding="utf-8") as f:
        return json.load(f)


risk_scores = load_json("risk_scores.json")["scores"]
dependencies = load_json("application_dependency.json")["applications"]
recommendations = load_json("recommendations.json")["recommendations"]
cmdb_assets = load_json("cmdb_assets.json")["assets"]


# ============================================================
# DATA PREP
# ============================================================
apps = [x["application"] for x in risk_scores]
scores = [x["score"] for x in risk_scores]

risk_colors = [
    "#ff3b30" if s >= 90 else
    "#ff9500" if s >= 70 else
    "#30d158"
    for s in scores
]


# ============================================================
# FIGURE
# ============================================================
fig = plt.figure(figsize=(26,16))
fig.patch.set_facecolor(BG)

fig.suptitle(
    "AI Migration Executive Control Center",
    fontsize=24,
    fontweight="bold",
    color="white",
    y=0.98
)


# ============================================================
# PANEL STYLE
# ============================================================
def style_panel(ax, title):
    ax.set_facecolor(PANEL)
    for spine in ax.spines.values():
        spine.set_color("#2a3441")
    ax.set_title(
        title,
        color=TEXT,
        fontsize=12,
        fontweight="bold",
        pad=8
    )


# ============================================================
# LIVE ALERT STRIP
# ============================================================
ax0 = plt.subplot2grid((5,4),(0,0), colspan=4)
style_panel(ax0, "Live Migration Risk Feed")

ax0.set_xlim(0,24)
ax0.set_ylim(0,3)
ax0.axis("off")

x = 1

for idx, app in enumerate(apps):

    score = scores[idx]

    color = (
        "#ff3b30" if score >= 90 else
        "#ff9500" if score >= 70 else
        "#30d158"
    )

    ax0.add_patch(Rectangle((x,1),3,1,color=color, alpha=0.95))

    ax0.text(
        x+0.25,
        1.35,
        f"{app} ({score})",
        color="black",
        fontsize=9,
        fontweight="bold"
    )

    x += 4


# ============================================================
# RISK PORTFOLIO
# ============================================================
ax1 = plt.subplot2grid((5,4),(1,0))
style_panel(ax1, "Migration Risk Portfolio")

ax1.bar(apps, scores, color=risk_colors)

ax1.tick_params(axis='x', rotation=20, colors='white', labelsize=8)
ax1.tick_params(axis='y', colors='white', labelsize=8)


# ============================================================
# DEPENDENCY
# ============================================================
ax2 = plt.subplot2grid((5,4),(1,1))
style_panel(ax2, "Dependency Criticality")

dep_counts = [len(x["dependencies"]) for x in dependencies]

ax2.bar(apps, dep_counts, color=ACCENT)

ax2.tick_params(axis='x', rotation=20, colors='white', labelsize=8)
ax2.tick_params(axis='y', colors='white', labelsize=8)


# ============================================================
# AI CONFIDENCE
# ============================================================
ax3 = plt.subplot2grid((5,4),(1,2))
style_panel(ax3, "AI Confidence")

confidence = 87

ax3.bar(["Confidence"], [confidence], color="#64d2ff")
ax3.set_ylim(0,100)
ax3.tick_params(colors='white', labelsize=8)


# ============================================================
# SEVERITY PANEL
# ============================================================
ax4 = plt.subplot2grid((5,4),(1,3))
style_panel(ax4, "Severity Status")

ax4.axis("off")

severity = """
CRITICAL : FinanceApp
WARNING  : BillingApp
READY    : ReportingApp
"""

ax4.text(
    0,
    1,
    severity,
    fontsize=11,
    fontweight="bold",
    verticalalignment="top",
    color="white"
)


# ============================================================
# CMDB
# ============================================================
ax5 = plt.subplot2grid((5,4),(2,0), colspan=2)
style_panel(ax5, "CMDB Asset Intelligence")

ax5.axis("off")

cmdb_lines = []

for asset in cmdb_assets[:4]:

    hostname = asset.get("hostname", asset.get("server","Unknown"))
    env = asset.get("environment","PROD")
    osname = asset.get("os","Linux")

    db = asset.get("database")
    if not db:
        db = "No DB mapped"

    cmdb_lines.append(f"{hostname} | {env} | {osname} | {db}")

ax5.text(
    0,
    1,
    "\n".join(cmdb_lines),
    fontsize=10,
    verticalalignment="top",
    color="white"
)


# ============================================================
# BLAST RADIUS
# ============================================================
ax6 = plt.subplot2grid((5,4),(2,2), colspan=2)
style_panel(ax6, "Blast Radius Impact Chain")

ax6.set_xlim(0,22)
ax6.set_ylim(0,5)
ax6.axis("off")

selected = dependencies[0]
nodes = [selected["application"]] + selected["dependencies"]

x = 1

for idx,node in enumerate(nodes):

    color = "#ff3b30" if idx == 0 else "#3a7ca5"
    text_color = "white" if idx == 0 else "black"

    ax6.add_patch(Rectangle((x,2),2.2,1,color=color))

    ax6.text(
        x+0.15,
        2.4,
        node,
        color=text_color,
        fontsize=8,
        fontweight="bold"
    )

    if idx > 0:
        ax6.arrow(
            x-1,
            2.5,
            0.8,
            0,
            head_width=0.1,
            head_length=0.2,
            color="white"
        )

    x += 3


# ============================================================
# ARCHITECTURE
# ============================================================
ax7 = plt.subplot2grid((5,4),(3,0), colspan=2)
style_panel(ax7, "Target Architecture Path")

ax7.set_xlim(0,20)
ax7.set_ylim(0,5)
ax7.axis("off")

layers = ["Web Tier","API Tier","Database","Landing Zone"]

x = 1

for node in layers:

    ax7.add_patch(Rectangle((x,2),3,1,color="#1f2937"))

    ax7.text(
        x+0.5,
        2.4,
        node,
        color="white",
        fontsize=8,
        fontweight="bold"
    )

    if x > 1:
        ax7.arrow(
            x-1,
            2.5,
            0.8,
            0,
            head_width=0.1,
            head_length=0.2,
            color="white"
        )

    x += 4


# ============================================================
# CLOUD TARGET
# ============================================================
ax8 = plt.subplot2grid((5,4),(3,2))
style_panel(ax8, "Cloud Target Decision")

ax8.axis("off")

cloud = """
FinanceApp → Azure
BillingApp → AWS
ReportingApp → IBM Cloud
"""

ax8.text(
    0,
    1,
    cloud,
    fontsize=10,
    verticalalignment="top",
    color="white"
)


# ============================================================
# AI RECOMMENDATIONS
# ============================================================
ax9 = plt.subplot2grid((5,4),(3,3))
style_panel(ax9, "AI Recommendations")

ax9.axis("off")

rec_lines = []

for r in recommendations[:4]:
    app = r["application"]

    for item in r["recommendations"][:1]:
        rec_lines.append(f"{app}: {item}")

ax9.text(
    0,
    1,
    "\n".join(rec_lines),
    fontsize=9,
    verticalalignment="top",
    color="white"
)


# ============================================================
# TIMELINE
# ============================================================
ax10 = plt.subplot2grid((5,4),(4,0), colspan=2)
style_panel(ax10, "Migration Execution Timeline")

timeline_apps = [
    "CitrixAccessGateway",
    "HRPortal",
    "ReportingApp",
    "BillingApp",
    "FinanceApp"
]

timeline_wave = [1,2,3,4,5]

ax10.barh(
    timeline_apps,
    timeline_wave,
    color=["#30d158","#30d158","#ffd60a","#ff9500","#ff3b30"]
)

ax10.tick_params(colors='white', labelsize=8)


# ============================================================
# NARRATIVE
# ============================================================
ax11 = plt.subplot2grid((5,4),(4,2), colspan=2)
style_panel(ax11, "AI Migration Narrative")

ax11.axis("off")

narrative = """
Traditional tools stop at findings.

This control center converts:
CMDB + Dependency + Architecture + Migration Wave + AI Decision

into migration execution intelligence.
"""

ax11.text(
    0,
    1,
    narrative,
    fontsize=10,
    verticalalignment="top",
    color="white"
)


plt.subplots_adjust(
    left=0.03,
    right=0.98,
    top=0.94,
    bottom=0.04,
    hspace=0.45,
    wspace=0.25
)

plt.show()